# Setup

In [ ]:
# Colab-specific setup
import pathlib

if 'google.colab' not in str(get_ipython()):
    base_folder = pathlib.Path('../../')
else:
    base_folder = pathlib.Path('/content/drive/MyDrive/Vision/')
    
    # install extra notebook dependencies in Colab
    ! uv pip install 'watermark==2.4.*'

    # mount colab folder
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
# show library versions
import watermark

# scikit-image also installs imageio and pillow
print(watermark.watermark(packages='skimage,imageio,PIL'))

# Image conversion
1. select **10** random images from the **small** dataset
2. convert images to gray
3. save images to a **prepared** dataset

In [ ]:
import io
import imageio.v3 as iio
import random
import skimage
import zipfile

In [ ]:
random.seed(0) # ensure reproducibility

original = base_folder / 'datasets' / '1-coins-small.zip'
prepared = base_folder / 'prepared' / '1-coins-small-10.zip'

with zipfile.ZipFile(original, 'r') as src, \
     zipfile.ZipFile(prepared, 'w') as dst:
    namelist = [n for n in src.namelist() if n.endswith('.jpg')]
    namelist = random.sample(namelist, k=10)

    for input_name in sorted(namelist):
        data = io.BytesIO(src.read(input_name))
        image = skimage.io.imread(data, as_gray=True)
        image = skimage.util.img_as_ubyte(image)

        output_name = pathlib.Path(input_name).with_suffix('.jpg').name

        # save image (for debugging)
        #skimage.io.imsave(output_name, image)
        #print('saved', output_name)

        data = io.BytesIO()
        iio.imwrite(data, image, extension='.jpg')
        dst.writestr(output_name, data.getbuffer())
        print('wrote', output_name)